In [1]:
import pandas as pd
import os
import numpy as np
import folium
import matplotlib.pyplot as plt
import textwrap

from matplotlib.backends.backend_pdf import PdfPages
from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime
from pyproj import Transformer

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

In [2]:
# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zur Datenbank, die verwendet werden soll 
db_root = (
    base_dir.parent.parent 
    / "1_Acquisition"
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken" / "Nach_Acquisition" / "Komplette_Datenbank"
)

# Alle Unterordner lesen, Ordner mit "neustem" Datum setzen
timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

# Komplette Datenbank als CSV
database_path = latest_folder / "Komplette_Datenbank.csv"

print("Verwendeter Datenbankpfad:")
print(database_path)

Verwendeter Datenbankpfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\Komplette_Datenbank\2026-01-05_01-03-41\Komplette_Datenbank.csv


In [3]:
# Ordner "Datenanalyse"
analysis_root = base_dir / "Datenanalyse"
analysis_root.mkdir(exist_ok=True)

# Zeitstempel erzeugen
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Neuen Ordner mit Zeitstempel erzeugen
output_dir = analysis_root / timestamp
output_dir.mkdir(parents=True, exist_ok=False)

print("Erzeugter Output-Ordner:")
print(output_dir)

Erzeugter Output-Ordner:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.1_Explorative-Datenanalyse\Datenanalyse\2026-01-05_01-05-15


<div class="alert alert-info">
    <b>Definition von ersten Funktionen zur Histogramm-Erstellung für Datenverteilung</b><br>
    - Balkendiagramm mit nicht-leeren Werten pro Spalte<br>
    - Eckdaten + Vollständigkeit in 10er-Schritten<br>
    - Histogramme für jede Spalte
</div>

In [4]:
# Erzwingt numerisch, wo möglich | Strings werden NaN, falls nicht konvertierbar
def _try_force_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

# Erzweingt datetime, wo möglich
def _try_force_datetime(s: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    if s.dtype == "object":
        parsed = pd.to_datetime(s, errors="coerce")
        if parsed.notna().mean() > 0.5:
            return parsed
    return s

In [5]:
# Erstellt eine Seite pro Datenspalte mit passendem Darstellungstyp
def _column_page(ax, col: str, s: pd.Series, sample_for_categoricals: int = 200000):
    n_total = len(s)
    n_nonnull = int(s.notna().sum())
    n_null = n_total - n_nonnull

    # Figure-Layout
    ax.axis("off")
    fig = ax.figure
    fig.suptitle(f"Spalte: {col}", fontsize=16, y=0.98)

    # Info-Text
    info_text = (
        f"Gesamt: {n_total}\n"
        f"Nicht leer: {n_nonnull} ({n_nonnull/n_total:.1%})\n"
        f"Leer: {n_null} ({n_null/n_total:.1%})\n"
        f"Original-Typ: {s.dtype}"
    )
    ax.text(0.02, 0.92, info_text, va="top", ha="left",
            fontsize=11, family="monospace")

    # Achsenbereiche für Plots
    plot_ax1 = fig.add_axes([0.08, 0.10, 0.84, 0.35])
    plot_ax2 = fig.add_axes([0.08, 0.55, 0.84, 0.35])

    # Überprüfung ob numerisch
    s_num = _try_force_numeric(s)
    if s_num.notna().sum() > 0:
        s_clean = s_num.dropna()

        # sinnvolle X-Skala über Quantile
        q1, q99 = s_clean.quantile([0.01, 0.99])
        if np.isfinite(q1) and np.isfinite(q99) and q1 < q99:
            x_min, x_max = q1, q99
        else:
            x_min, x_max = s_clean.min(), s_clean.max()

        # Freedman–Diaconis-Bins
        bins = "fd"

        # Histogramm-Plotting
        plot_ax2.hist(s_clean, bins=bins, range=(x_min, x_max))
        plot_ax2.set_title("Numerische Verteilung (Histogramm)")
        plot_ax2.set_xlabel(col)
        plot_ax2.set_ylabel("Häufigkeit")

        # optional Log-Y bei sehr schiefer Verteilung
        med = s_clean.median()
        if med > 0 and s_clean.max() > 50 * med:
            plot_ax2.set_yscale("log")

        count_vals = len(s_clean)
        perc_vals = count_vals / n_total * 100

        # einfache Statistik als Ausgabe auf jeweiliger Seite
        stats_text = (
            "Basisstatistik (bereinigte Werte):\n"
            f"  Anzahl Werte: {count_vals} ({perc_vals:.2f}% aller Zeilen)\n"
            f"  Min:   {s_clean.min():.6g}\n"
            f"  1%:    {s_clean.quantile(0.01):.6g}\n"
            f"  25%:   {s_clean.quantile(0.25):.6g}\n"
            f"  Median:{s_clean.median():.6g}\n"
            f"  75%:   {s_clean.quantile(0.75):.6g}\n"
            f"  99%:   {s_clean.quantile(0.99):.6g}\n"
            f"  Max:   {s_clean.max():.6g}\n"
            f"  Mittel:{s_clean.mean():.6g}\n"
            f"  Std:   {s_clean.std(ddof=1):.6g}"
        )
        plot_ax1.axis("off")
        plot_ax1.text(0.01, 0.95, stats_text,
                      va="top", ha="left",
                      fontsize=11, family="monospace")
        return

# -----------------------------------------

    # Force Datetime
    s_dt = _try_force_datetime(s)
    if pd.api.types.is_datetime64_any_dtype(s_dt) and s_dt.notna().sum() > 0:
        s_dt_clean = s_dt.dropna()

        plot_ax2.hist(s_dt_clean.view("int64"), bins=30)
        plot_ax2.set_title("Zeitverteilung (Histogramm)")
        plot_ax2.set_ylabel("Häufigkeit")

        try:
            by_year = s_dt_clean.dt.to_period("Y").value_counts().sort_index()
            plot_ax1.bar(by_year.index.astype(str), by_year.values)
            plot_ax1.set_title("Counts pro Jahr")
            plot_ax1.set_ylabel("Häufigkeit")
            for label in plot_ax1.get_xticklabels():
                label.set_rotation(45)
                label.set_ha("right")
        except Exception:
            plot_ax1.text(0.5, 0.5, "Keine sinnvolle Jahres-Aggregation",
                          ha="center", va="center")
        return

    # Katgeorische mit Texten
    s_cat = s.dropna().astype(str)
    if len(s_cat) > 0:
        if len(s_cat) > sample_for_categoricals:
            s_cat = s_cat.sample(sample_for_categoricals, random_state=0)

        vc = s_cat.value_counts()
        top_n = 20
        vc_top = vc.head(top_n)

        plot_ax2.bar(vc_top.index, vc_top.values)
        plot_ax2.set_title(f"Kategorische Verteilung (Top {top_n})")
        plot_ax2.set_xlabel("Kategorie")
        plot_ax2.set_ylabel("Häufigkeit")
        plot_ax2.tick_params(axis="x", rotation=45)

        rest = vc.iloc[top_n:].sum()
        if rest > 0:
            plot_ax1.pie(
                [vc_top.sum(), rest],
                labels=[f"Top {top_n}", "Rest"],
                autopct=lambda p: f"{p:.1f}%"
            )
            plot_ax1.set_title("Top vs. Rest")
        else:
            plot_ax1.text(0.5, 0.5, "Keine weiteren Kategorien",
                          ha="center", va="center")
        return


    plot_ax1.text(0.5, 0.5, "Keine Werte", ha="center", va="center")
    plot_ax2.text(0.5, 0.5, "–", ha="center", va="center")

In [6]:
# ----------------- Spalten-Konfiguration -----------------

PORTRAIT_HBAR_COLS = {c.strip() for c in [
    "location",
    "rock_type",
    "stratigraphic_period",
    "Database_number",
]}

LANDSCAPE_NUMERIC_COLS = {c.strip() for c in [
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
]}

SKIP_COLS = {c.strip() for c in [
    "well_or_spring_name",
    "Database_name",
]}

In [7]:
def _column_page(
    ax,
    col: str,
    s: pd.Series,
    sample_for_categoricals: int = 200000,
    force_categorical: bool = False,
):
    """
    Zeichnet eine einzelne PDF-Seite für die Spalte `col`.

    # numerisch: Histogramm (lineare Skala, adaptiver Cutoff)
    # Datum/Zeit: Häufigkeit pro Jahr
    # kategorial: horizontales Balkendiagramm, Labels umbrochen
    # force_categorical=True: Spalte wird unabhängig vom Datentyp als Kategorie behandelt

    Die Info-Box enthält immer:
    # Gesamt, Nicht leer, Leer
    und bei numerischen Spalten zusätzlich:
    # Minimum, Maximum, Mittelwert, Standardabweichung
    """
    from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype

    ax.clear()
    ax.set_title(f"Spalte: {col}", fontsize=14, pad=12)

    n_total = len(s)
    n_nonnull = int(s.notna().sum())
    n_null = n_total - n_nonnull

    # --------------- Fehlerbehandlung, falls komplett leer
    if n_nonnull == 0:
        info = (
            f"Gesamt: {n_total}\n"
            f"Nicht leer: {n_nonnull} (0.0%)\n"
            f"Leer: {n_null} (100.0%)"
        )
        ax.text(
            0.01, 0.99, info,
            ha="left", va="top", transform=ax.transAxes,
            fontsize=9, family="monospace",
            bbox=dict(boxstyle="round", alpha=0.1),
        )
        ax.text(0.5, 0.5, "Keine Werte", ha="center", va="center", fontsize=12)
        ax.set_xticks([]); ax.set_yticks([])
        return

    s_nonnull = s.dropna()
    col_key = str(col).strip()

    # ------------- Typ-Flags ----------------
    from pandas.api.types import is_numeric_dtype as _is_numeric_dtype
    from pandas.api.types import is_datetime64_any_dtype as _is_datetime64_any_dtype

    is_num_col = _is_numeric_dtype(s_nonnull) or (col_key in LANDSCAPE_NUMERIC_COLS)
    is_dt_col = _is_datetime64_any_dtype(s_nonnull) or np.issubdtype(s_nonnull.dtype, np.datetime64)

    # -------- Info-Box mit Statistiken --------
    info = (
        f"Gesamt: {n_total}\n"
        f"Nicht leer: {n_nonnull} ({n_nonnull / max(n_total, 1):.1%})\n"
        f"Leer: {n_null} ({n_null / max(n_total, 1):.1%})"
    )

    if is_num_col:
        data_stats = pd.to_numeric(s_nonnull, errors="coerce").dropna()
        if not data_stats.empty:
            info += (
                f"\nMin: {data_stats.min():.3g}"
                f"\nMax: {data_stats.max():.3g}"
                f"\nMittelwert: {data_stats.mean():.3g}"
                f"\nStd-Abw.: {data_stats.std():.3g}"
            )

    ax.text(
        0.01, 0.99, info,
        ha="left", va="top", transform=ax.transAxes,
        fontsize=9, family="monospace",
        bbox=dict(boxstyle="round", alpha=0.1),
    )

    # ---------------------- Katgeroisches erzwingen ----------------------
    if force_categorical and not is_num_col:
        s_cat = s_nonnull.astype(str)
        if len(s_cat) > sample_for_categoricals:
            s_cat = s_cat.sample(sample_for_categoricals, random_state=0)

        vc = s_cat.value_counts().head(50)
        if vc.empty:
            ax.text(0.5, 0.5, "Keine auswertbaren Werte",
                    ha="center", va="center", fontsize=12)
            return

        y = np.arange(len(vc))
        labels = [textwrap.fill(v, 30) for v in vc.index]

        ax.barh(y, vc.values)
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.set_xlabel("Häufigkeit")
        ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        return

    # ---------------------- Numerische Werte identifizieren ----------------------
    if is_num_col:
        data = pd.to_numeric(s_nonnull, errors="coerce").dropna()
        if data.empty:
            ax.text(0.5, 0.5, "Keine numerischen Werte",
                    ha="center", va="center", fontsize=12)
            return

        real_min = float(data.min())
        real_max = float(data.max())

        # ------------ Perzentile für Cutoff, bessere Darstellung -------------
        p98 = float(np.percentile(data, 98))
        p99 = float(np.percentile(data, 99))

        if real_max > p99 * 10:
            cutoff = p98
        else:
            cutoff = p99

        cutoff = max(cutoff, real_min)

        # ----------- Bins abhängig von n --------------
        n = len(data)
        if n < 30:
            bins = max(5, n // 2)
        else:
            bins = min(60, int(np.sqrt(n)))

        ax.hist(data, bins=bins, range=(real_min, cutoff))
        ax.set_xlim(real_min, cutoff)
        ax.set_xlabel(col)
        ax.set_ylabel("Häufigkeit")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        return

    # ---------------------- Datum erzeugen ----------------------
    if is_dt_col:
        dt = pd.to_datetime(s_nonnull, errors="coerce").dropna()
        if dt.empty:
            ax.text(0.5, 0.5, "Keine gültigen Datumswerte",
                    ha="center", va="center", fontsize=12)
            return

        counts = dt.dt.to_period("Y").value_counts().sort_index()
        ax.bar(counts.index.astype(str), counts.values)
        ax.set_xlabel("Jahr")
        ax.set_ylabel("Anzahl")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        return

    # ---------------------- Kategorische Auswertung ----------------------
    s_cat = s_nonnull.astype(str)
    vc = s_cat.value_counts().head(20)

    if vc.empty:
        ax.text(0.5, 0.5, "Keine auswertbaren Werte",
                ha="center", va="center", fontsize=12)
        return

    y = np.arange(len(vc))
    labels = [textwrap.fill(v, 25) for v in vc.index]

    ax.barh(y, vc.values)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("Häufigkeit")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()

In [8]:
def create_data_distribution_pdf_from_paths(
    database_path: Path,
    output_dir: Path,
    pdf_name: str = "Komplette_Datenbank_Histogramme.pdf",
    sample_for_categoricals: int = 200000,
) -> Path:

    database_path = Path(database_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = output_dir / pdf_name

    if not database_path.exists():
        raise FileNotFoundError(f"Input-Datei existiert nicht: {database_path}")

    print("Starte PDF-Erstellung…")
    print(f"Lade Datenbank: {database_path}")
    df = pd.read_csv(database_path, low_memory=False)

    n_rows, n_cols = df.shape
    print(f"Daten geladen: {n_rows} Zeilen, {n_cols} Spalten")

    nonnull_counts = df.notna().sum().sort_values(ascending=False)

    # Zeilen-Vollständigkeit
    row_comp = df.notna().mean(axis=1) * 100
    complete_rows = int((row_comp == 100).sum())

    bins = np.arange(0, 110, 10)
    labels = [f"{bins[i]}–{bins[i+1]}%" for i in range(len(bins) - 1)]
    binned = pd.cut(row_comp, bins=bins, labels=labels, include_lowest=True)
    bin_counts = binned.value_counts().reindex(labels, fill_value=0)
    bin_percent = (bin_counts / n_rows * 100).round(2)

    with PdfPages(pdf_path) as pdf:

        # -------- Seite 1: Spaltenvollständigkeit --------
        print("Seite 1: Spalten-Vollständigkeit")
        fig, ax = plt.subplots(figsize=(8.3, 11.7))
        ax.barh(nonnull_counts.index.astype(str), nonnull_counts.values)
        ax.set_title("Anzahl nicht-leerer Werte pro Spalte")
        ax.set_xlabel("Nicht-leere Werte")
        ax.set_ylabel("Spalte")
        ax.set_xlim(0, n_rows)
        ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # -------- Seite 2: Eckdaten / Zeilen-Vollständigkeit --------
        print("Seite 2: Eckdaten / Zeilen-Vollständigkeit")
        fig, ax = plt.subplots(figsize=(8.3, 11.7))
        ax.axis("off")
        ax.set_title("Eckdaten / Zeilen-Vollständigkeit", fontsize=16, pad=20)

        txt = (
            f"Zeilen: {n_rows}\n"
            f"Spalten: {n_cols}\n"
            f"Vollständige Zeilen (100% befüllt): {complete_rows} "
            f"({complete_rows / n_rows:.2%})\n\n"
            "Verteilung der Zeilen-Vollständigkeit (10%-Schritte):\n"
        )
        for l in labels:
            txt += f"• {l}: {bin_percent[l]}% aller Datenzeilen\n"

        ax.text(
            0.05, 0.95, txt,
            ha="left", va="top",
            fontsize=12, family="monospace",
        )

        pdf.savefig(fig)
        plt.close(fig)

        # -------- Seite 3-x: Spalten -----------
        print("Spaltenseiten")
        for i, col in enumerate(df.columns, start=1):
            col_key = str(col).strip()

            if col_key in SKIP_COLS:
                print(f"   → {i}/{len(df.columns)}: {col} (übersprungen)")
                continue

            if col_key in LANDSCAPE_NUMERIC_COLS:
                figsize = (11.7, 8.3)  # -------- Querformat ----------
            else:
                figsize = (8.3, 11.7)  # -------- Hochformat ----------

            print(f"   → {i}/{len(df.columns)}: {col} (figsize={figsize})")
            fig, ax = plt.subplots(figsize=figsize)

            _column_page(
                ax,
                col,
                df[col],
                sample_for_categoricals=sample_for_categoricals,
                force_categorical=(col_key in PORTRAIT_HBAR_COLS),
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"PDF fertig: {pdf_path.resolve()}")
    return pdf_path

In [9]:
pdf_path = create_data_distribution_pdf_from_paths(
    database_path=database_path,
    output_dir=output_dir,
    pdf_name="Komplette_Datenbank_Histogramme.pdf"
)

print("PDF gespeichert unter:", pdf_path)

Starte PDF-Erstellung…
Lade Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\Komplette_Datenbank\2026-01-05_01-03-41\Komplette_Datenbank.csv


Daten geladen: 175099 Zeilen, 31 Spalten
Seite 1: Spalten-Vollständigkeit


Seite 2: Eckdaten / Zeilen-Vollständigkeit
Spaltenseiten
   → 1/31: location (figsize=(8.3, 11.7))


   → 2/31: well_or_spring_name (übersprungen)
   → 3/31: WGS84_latitude (figsize=(11.7, 8.3))


   → 4/31: WGS84_longitude (figsize=(11.7, 8.3))


...

   → 25/31: Mn_in_mmol/L (figsize=(11.7, 8.3))
   → 26/31: F_in_umol/L (figsize=(11.7, 8.3))


   → 27/31: NO3_in_mmol/L (figsize=(11.7, 8.3))


   → 28/31: H2SiO3_in_umol/L (figsize=(11.7, 8.3))


   → 29/31: HS_in_mmol/L (figsize=(11.7, 8.3))
   → 30/31: Database_number (figsize=(8.3, 11.7))


   → 31/31: Database_name (übersprungen)
PDF fertig: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.1_Explorative-Datenanalyse\Datenanalyse\2026-01-05_01-05-15\Komplette_Datenbank_Histogramme.pdf
PDF gespeichert unter: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.1_Explorative-Datenanalyse\Datenanalyse\2026-01-05_01-05-15\Komplette_Datenbank_Histogramme.pdf


<div class="alert alert-info">
    <b>Als Nächstes wird manuell nach Ausreißern im Datensastz gesucht</b><br>
</div>